### AITW-Clearwood_stiffness_model

An hybrid multiscale modelling framework that combines analytical continuum micromechanics with a numerical finite-element unit-cell approach, within the scope of linear elasticity, establishing quantitative links between its multiscale microstructure and macroscopic stiffness.

#### Links

- <a href="https://github.com/AI-TranspWood/Clearwood_share" target="_blank" rel="noopener noreferrer">Source code</a>
- <a href="https://www.eessi.io/docs/available_software/detail/AITW-Clearwood_stiffness_model" target="_blank" rel="noopener noreferrer">EESSI</a>

#### References

- <a href="https://doi.org/10.1007/s00226-026-01789-0" target="_blank" rel="noopener noreferrer">Thirion, Arthur, et al.  Wood Science and Technology 60.5 (2026): 80.</a>

In [ ]:
import os
# Proper environment initialization for julia packages relies on these paths to not be set to an empty string
try:
    del os.environ['JULIA_DEPOT_PATH']
    del os.environ['JULIA_LOAD_PATH']
except Exception:
    pass

from utilities import initialize_with_module

await initialize_with_module('AITW-Clearwood_stiffness_model/0.1.0-Julia-1.12.2', '2025.06')

In [ ]:
import re
import os
import json
import subprocess
import tempfile
import ipywidgets as wdg
from IPython.display import display

import utilities
from common import CLEARWOOD_STIFFNESS_OUTDIR

title = wdg.HTML(value="<h3>Run inference</h3>")


config_file = wdg.Text(
    value='../example-inputs/clearwood/config.json',
    description='Config File:',
    layout=wdg.Layout(width='50%'),
)
output_file = wdg.Text(
    value=os.path.join(CLEARWOOD_STIFFNESS_OUTDIR, 'results.json'),
    description='Output File:',
    layout=wdg.Layout(width='50%'),
)

#############################################################################################################
# Earlywood (EW) Wood Cell Geometry
ew_tension = wdg.Text(
    description='EW Tension:',
    placeholder='Typical: 0.5 - 0.99',
    layout=wdg.Layout(width='50%')
)
ew_tension_tooltip = wdg.Button(
    description='?',
    tooltip=(
        "Governs the transition from smooth, rounded cell corners at vanishing τ to increasingly sharp, kink-like "
        "features (Typical range: 0.5 – 0.99)."
    ),
    disabled=True,
    button_style='info'
)
ew_lumen_aspect_ratio = wdg.Text(
    description='EW Lumen Aspect Ratio:',
    placeholder='Typical: 0.9 - 1.5',
    layout=wdg.Layout(width='50%')
)
ew_lumen_aspect_ratio_tooltip = wdg.Button(
    description='?',
    tooltip=(
        "The ratio of the lumen radius (r_R/r_T) in the radial direction r_R to that in the tangential direction r_T (Typical range: 0.9 – 1.5)."
    ),
    disabled=True,
    button_style='info'
)
ew_tangential_offset = wdg.Text(
    description='EW Tangential Offset:',
    placeholder='Typical: 0.7 - 1.0',
    layout=wdg.Layout(width='50%')
)
ew_tangential_offset_tooltip = wdg.Button(
    description='?',
    tooltip=(
        "Quantifies the continuity of the tangential webs. An offset of 1 corresponds to a standard softwood structure (every second row shifted by half a cell), whereas an offset of 0 results in continuous tangential webs (Typical range: 0.7 – 1.0)."
    ),
    disabled=True,
    button_style='info'
)
ew_interior_angle = wdg.Text(
    description='EW Interior Angle:',
    placeholder='Typical: 150 - 180',
    layout=wdg.Layout(width='50%')
)
ew_interior_angle_tooltip = wdg.Button(
    description='?',
    tooltip=(
        "Defines the obtuse angle between the radial webs; cell walls become more continuous as the angle approaches 180° (Typical range: 150° – 180°)."
    ),
    disabled=True,
    button_style='info'
)
ew_tension_tooltip.layout.width = '30px'
ew_lumen_aspect_ratio_tooltip.layout.width = '30px'
ew_tangential_offset_tooltip.layout.width = '30px'
ew_interior_angle_tooltip.layout.width = '30px'
ew_box = wdg.VBox([
    wdg.HBox([ew_tension, ew_tension_tooltip]),
    wdg.HBox([ew_lumen_aspect_ratio, ew_lumen_aspect_ratio_tooltip]),
    wdg.HBox([ew_tangential_offset, ew_tangential_offset_tooltip]),
    wdg.HBox([ew_interior_angle, ew_interior_angle_tooltip]),
])

#############################################################################################################
# Latewood (LW) Wood Cell Geometry
lw_tension = wdg.Text(
    description='LW Tension:',
    placeholder='Typical: 0.3 - 0.99',
    layout=wdg.Layout(width='50%')
)
lw_tension_tooltip = wdg.Button(
    description='?',
    tooltip=(
        "Governs the transition from smooth, rounded cell corners to sharp features (Typical range: 0.3 – 0.99)."
    ),
    disabled=True,
    button_style='info'
)
lw_lumen_aspect_ratio = wdg.Text(
    description='LW Lumen Aspect Ratio:',
    placeholder='Typical: 0.6 - 1.1',
    layout=wdg.Layout(width='50%')
)
lw_lumen_aspect_ratio_tooltip = wdg.Button(
    description='?',
    tooltip=(
        "Ratio of radial radius to tangential radius (r_R/r_T) (Typical range: 0.6 – 1.1)."
    ),
    disabled=True,
    button_style='info'
)
lw_tangential_offset = wdg.Text(
    description='LW Tangential Offset:',
    placeholder='Typical: 0.7 - 1.0',
    layout=wdg.Layout(width='50%')
)
lw_tangential_offset_tooltip = wdg.Button(
    description='?',
    tooltip=(
        "Continuity of tangential webs (1 for standard softwood, 0 for continuous) (Typical range: 0.7 – 1.0)."
    ),
    disabled=True,
    button_style='info'
)
lw_interior_angle = wdg.Text(
    description='LW Interior Angle:',
    placeholder='Typical: 150 - 180',
    layout=wdg.Layout(width='50%')
)
lw_interior_angle_tooltip = wdg.Button(
    description='?',
    tooltip=(
        "Defines the obtuse angle between radial webs (Typical range: 150° – 180°)."
    ),
    disabled=True,
    button_style='info'
)
lw_tension_tooltip.layout.width = '30px'
lw_lumen_aspect_ratio_tooltip.layout.width = '30px'
lw_tangential_offset_tooltip.layout.width = '30px'
lw_interior_angle_tooltip.layout.width = '30px'
lw_box = wdg.VBox([
    wdg.HBox([lw_tension, lw_tension_tooltip]),
    wdg.HBox([lw_lumen_aspect_ratio, lw_lumen_aspect_ratio_tooltip]),
    wdg.HBox([lw_tangential_offset, lw_tangential_offset_tooltip]),
    wdg.HBox([lw_interior_angle, lw_interior_angle_tooltip]),
])

#############################################################################################################
# Clearwood Density
ew_density = wdg.Text(
    description='EW Density:',
    placeholder='Typical: 0.3 - 0.5',
    layout=wdg.Layout(width='50%')
)
ew_density_tooltip = wdg.Button(
    description='?',
    tooltip=(
        "Earlywood density in g/cm^3 (Typical range: 0.3 – 0.5)."
    ),
    disabled=True,
    button_style='info'
)
lw_density = wdg.Text(
    description='LW Density:',
    placeholder='Typical: 0.6 - 1.0',
    layout=wdg.Layout(width='50%')
)
lw_density_tooltip = wdg.Button(
    description='?',
    tooltip=(
        "Latewood density in g/cm^3 (Typical range: 0.6 – 1.0)."
    ),
    disabled=True,
    button_style='info'
)
lw_volume_fraction = wdg.Text(
    description='LW Volume Fraction:',
    placeholder='Typical: 0.05 - 0.5',
    layout=wdg.Layout(width='50%')
)
lw_volume_fraction_tooltip = wdg.Button(
    description='?',
    tooltip=(
        "Relative amount of the LW phase in clearwood (Typical range: 0.05 – 0.5)."
    ),
    disabled=True,
    button_style='info'
)
ew_density_tooltip.layout.width = '30px'
lw_density_tooltip.layout.width = '30px'
lw_volume_fraction_tooltip.layout.width = '30px'
ew_density_box = wdg.VBox([
    wdg.HBox([ew_density, ew_density_tooltip]),
    wdg.HBox([lw_density, lw_density_tooltip]),
    wdg.HBox([lw_volume_fraction, lw_volume_fraction_tooltip]),
])

#############################################################################################################
# Ray Properties
ray_volume_fraction = wdg.Text(
    description='Ray Volume Fraction:',
    placeholder='Typical: 0.04 - 0.17',
    layout=wdg.Layout(width='50%')
)
ray_volume_fraction_tooltip = wdg.Button(
    description='?',
    tooltip=(
        "The relative volume occupied by wood rays (Typical range: 0.04 – 0.17)."
    ),
    disabled=True,
    button_style='info'
)
ray_aspect_ratio = wdg.Text(
    description='Ray Aspect Ratio:',
    placeholder='Typical: 1.0 - 29.0',
    layout=wdg.Layout(width='50%')
)
ray_aspect_ratio_tooltip = wdg.Button(
    description='?',
    tooltip=(
        "The ratio of ray height to width (Typical range: 1.0 – 29.0)."
    ),
    disabled=True,
    button_style='info'
)
ray_volume_fraction_tooltip.layout.width = '30px'
ray_aspect_ratio_tooltip.layout.width = '30px'
ray_box = wdg.VBox([
    wdg.HBox([ray_volume_fraction, ray_volume_fraction_tooltip]),
    wdg.HBox([ray_aspect_ratio, ray_aspect_ratio_tooltip]),
])

#############################################################################################################
# Cellulose Microfibril Properties
ew_to_lw_mfa_ratio = wdg.Text(
    description='EW to LW MFA Ratio:',
    placeholder='Typical: 0.3 - 1.2',
    layout=wdg.Layout(width='50%')
)
ew_to_lw_mfa_ratio_tooltip = wdg.Button(
    description='?',
    tooltip=(
        "The ratio (LW/EW) of MFA between the two phases (Typical range: 0.3 – 1.2)."
    ),
    disabled=True,
    button_style='info'
)
ew_mfa = wdg.Text(
    description='EW Microfibril Angle (MFA):',
    placeholder='Typical: 8.0 - 30.0',
    layout=wdg.Layout(width='50%')
)
ew_mfa_tooltip = wdg.Button(
    description='?',
    tooltip=(
        "The orientation of cellulose microfibrils in the EW cell wall (Typical range: 8.0 – 30.0)."
    ),
    disabled=True,
    button_style='info'
)
ew_to_lw_mfa_ratio_tooltip.layout.width = '30px'
ew_mfa_tooltip.layout.width = '30px'
cellulose_box = wdg.VBox([
    wdg.HBox([ew_to_lw_mfa_ratio, ew_to_lw_mfa_ratio_tooltip]),
    wdg.HBox([ew_mfa, ew_mfa_tooltip]),
])

accordion1 = wdg.Accordion(children=[ew_box], selected_index=None)
accordion2 = wdg.Accordion(children=[lw_box], selected_index=None)
accordion3 = wdg.Accordion(children=[ew_density_box], selected_index=None)
accordion4 = wdg.Accordion(children=[ray_box], selected_index=None)
accordion5 = wdg.Accordion(children=[cellulose_box], selected_index=None)

accordion1.set_title(0, 'Earlywood (EW) Wood Cell Geometry')
accordion2.set_title(0, 'Latewood (LW) Wood Cell Geometry')
accordion3.set_title(0, 'Clearwood Density')
accordion4.set_title(0, 'Ray Properties')
accordion5.set_title(0, 'Cellulose Microfibril Properties')

accordion_box = wdg.VBox([accordion1, accordion2, accordion3, accordion4, accordion5])

def generate_input_json():
    config_file_path = config_file.value
    if os.path.exists(config_file_path):
        with open(config_file_path, 'r') as f:
            config_data = json.load(f)
    else:
        config_data = {
            "ew_cell": [None, None, None, None],
            "lw_cell": [None, None, None, None],
            "density": [None, None, None],
            "ray_props": [None, None],
            "fibre_props": [None, None]
        }

    if ew_tension.value != '':
        config_data['ew_cell'][0] = float(ew_tension.value)
    if ew_lumen_aspect_ratio.value != '':
        config_data['ew_cell'][1] = float(ew_lumen_aspect_ratio.value)
    if ew_tangential_offset.value != '':
        config_data['ew_cell'][2] = float(ew_tangential_offset.value)
    if ew_interior_angle.value != '':
        config_data['ew_cell'][3] = float(ew_interior_angle.value)
    
    if lw_tension.value != '':
        config_data['lw_cell'][0] = float(lw_tension.value)
    if lw_lumen_aspect_ratio.value != '':
        config_data['lw_cell'][1] = float(lw_lumen_aspect_ratio.value)
    if lw_tangential_offset.value != '':
        config_data['lw_cell'][2] = float(lw_tangential_offset.value)
    if lw_interior_angle.value != '':
        config_data['lw_cell'][3] = float(lw_interior_angle.value)

    if ew_density.value != '':
        config_data['density'][0] = float(ew_density.value)
    if lw_density.value != '':
        config_data['density'][1] = float(lw_density.value)
    if lw_volume_fraction.value != '':
        config_data['density'][2] = float(lw_volume_fraction.value)

    if ray_volume_fraction.value != '':
        config_data['ray_props'][0] = float(ray_volume_fraction.value)
    if ray_aspect_ratio.value != '':
        config_data['ray_props'][1] = float(ray_aspect_ratio.value)
    if ew_to_lw_mfa_ratio.value != '':
        config_data['fibre_props'][0] = float(ew_to_lw_mfa_ratio.value)
    if ew_mfa.value != '':
        config_data['fibre_props'][1] = float(ew_mfa.value)

    for key, value in config_data.items():
        if any(v is None for v in value):
            print(f"Warning: Some values for '{key}' are missing. Please fill in all fields.")
            return

    return config_data


def run_julia_code(data):
    """Generate a Julia input string from the provided data dictionary and run it."""
    with (
        tempfile.NamedTemporaryFile(delete=True, suffix='.json') as data_file,
        tempfile.NamedTemporaryFile(delete=True, suffix='.jl') as julia_file
    ):
        print(f"Temporary data file created at: {data_file.name}")
        with open(data_file.name, 'w') as fp:
            json.dump(data, fp)

        julia_input = f"""
        using Clearwood_stiffness_model

        models = load_models()
        x = make_input_from_json("{data_file.name}")
        res = combi_model(x, models)

        # Display the results
        println(res)
        """
        with open(julia_file.name, 'wb') as julia_file:
            julia_file.write(julia_input.encode('utf-8'))
        
        # Run the Julia code
        try:
            result = subprocess.run(
                ['julia', julia_file.name],
                capture_output=True,
                text=True,
                check=True
            )
        except subprocess.CalledProcessError as e:
            print("Error running Julia code:\n", e.stderr)

        output_keys = ['E_R', 'E_T', 'E_L', 'G_TL', 'G_RL', 'G_RT', 'nu_RT', 'nu_RL', 'nu_TL', 'nu_TR', 'nu_LR', 'nu_LT']
        output_vector = re.findall(r"[-+]?\d*\.\d+|\d+", result.stdout)
        if len(output_vector) != len(output_keys):
            print("Warning: The number of output values does not match the expected keys.")
            print("Expected keys:", output_keys)
            print("Output values:", output_vector)
        output_dict = dict(zip(output_keys, output_vector))
        print("Output:")
        print(json.dumps(output_dict, indent=2))

        if output_file.value:
            os.makedirs(os.path.dirname(output_file.value), exist_ok=True)
            output_data = {
                "input": data,
                "output": output_dict
            }
            if os.path.exists(output_file.value):
                print(f"WARNING: The output file '{output_file.value}' already exists and will be overwritten.")
            with open(output_file.value, 'w') as f:
                json.dump(output_data, f, indent=2)

            print(f"Results saved to {output_file.value}.")

output_area = wdg.Output()
button_event_observer: wdg.IntText = wdg.IntText(
    value=0, description='', disabled=True,
)

@utilities.widget_button_running(running_text="Evaluating...")
def on_run_button_clicked(b):
    with output_area:
        try:
            output_area.clear_output(wait=True)
            data = generate_input_json()
            if not manually_closed_output:
                output_accordion.selected_index = 0
            output_accordion.layout.border = "2px solid blue"
            run_julia_code(data)
            button_event_observer.value += 1
        except Exception as e:
            print(f"An error occurred: {e}")
            output_accordion.selected_index = 0
            output_accordion.layout.border = "2px solid red"
        else:
            output_accordion.layout.border = "2px solid green"
    
run_button = wdg.Button(description="Run Model", button_style='success')
run_button.on_click(on_run_button_clicked)

output_accordion = wdg.Accordion(children=[output_area], selected_index=None)
output_accordion.set_title(0, 'Output Logs')

manually_closed_output = False
def on_output_accordion_change(change):
    global manually_closed_output
    # Remove color of output area border when the user manually interacts with the output accordion
    output_accordion.layout.border = ""
    if change['new'] is None:
        manually_closed_output = True
output_accordion.observe(on_output_accordion_change, names='selected_index')

visualizer_accordion = utilities.output_visualizer_widget(
    title="Visualize Results",
    file_format='json',
    event_observer=button_event_observer,
    output_dir=CLEARWOOD_STIFFNESS_OUTDIR,
)

main_box = wdg.VBox([
    title,
    # reference,
    config_file,
    output_file,
    accordion_box,
    run_button,
    output_accordion,
])

main_box.add_class('widget-card')
title.add_class('widget-card-title')

display(main_box)
display(visualizer_accordion)

In [ ]:
import utilities
utilities.final_spacer()